# Telecom Customer Churn Prediction

**Author: Dishank Belwanshi**

This notebook explores the IBM Telco sample dataset, prepares the data, compares two classifiers and evaluates them on a held-out test set. Run cells from top to bottom.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().resolve()))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from src.data_loader import load_data
from src.train import prepare_data, make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


## 1. Load and inspect the data

In [ ]:
df = load_data()
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).head(10))


## 2. Target distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer churn distribution")
plt.xlabel("Churn")
plt.ylabel("Number of customers")
plt.tight_layout()
plt.show()
print((df["Churn"].value_counts(normalize=True) * 100).round(2))


## 3. Explore churn by contract and monthly charges

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=df, x="Contract", hue="Churn", ax=axes[0])
axes[0].set_title("Churn by contract type")
sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1])
axes[1].set_title("Monthly charges by churn status")
plt.tight_layout()
plt.show()


## 4. Prepare features and split the data

In [ ]:
X, y = prepare_data(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Training rows:", len(X_train), "| Test rows:", len(X_test))
print("Training churn rate:", round(y_train.mean(), 3))
print("Test churn rate:", round(y_test.mean(), 3))


## 5. Train and compare models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, class_weight="balanced",
        random_state=42, n_jobs=-1
    )
}
results = []
fitted_models = {}
for name, estimator in models.items():
    pipeline = make_pipeline(estimator)
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    results.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, probabilities)
    })
    fitted_models[name] = pipeline
    print("\\n", name)
    print(classification_report(y_test, predictions, target_names=["Retained", "Churned"]))
display(pd.DataFrame(results).sort_values("ROC-AUC", ascending=False))


## 6. Interpretation and next steps

Compare recall, precision, F1-score and ROC-AUC before choosing a model. A high recall may help identify more customers at risk, while precision reflects how many flagged customers actually churned in the test set. These findings are associations, not causal claims. See `REPORT.md` for limitations and responsible-use notes.